Customer Data Quality

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
def dqc_checks_customer(cust_table_name):

    df=spark.table(cust_table_name)

    clean_phone_digits = F.regexp_replace(F.col("phone"), r"[^0-9]", "")


    email_regex = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,4}$"
    contains_digits = r"[0-9]"
    contains_bad_symbols = r"[^A-Za-z\s\-\']"
    has_excess_spaces = r"\s{2,}"

    # Phone Validation Rules:
    # Rule A: Flags strings containing common spreadsheet error prefixes like #
    is_system_error = r"^#.*"
    
    # Rule B: Flags strings containing negative sign indicators
    is_negative_number = r"^-.*"
    
    # Rule C: Flags corporate extension suffixes like 'x123' or 'ext'
    has_phone_extension = r"(?i)x|ext"
    
    # Rule D: Standard valid format (Optional strict check if you want to enforce pure digits/dashes/parens only)
    # Allows only: numbers, spaces, dashes, periods, and parentheses
    valid_phone_characters = r"^[0-9\s\-\.\(\)]+$"


    df_with_id_count = df.withColumn(
        "_id_count", 
        F.count("CustomerID").over(Window.partitionBy("CustomerID"))
    )

    # 2. Build the structural DQ logic array
    error_conditions = [
        F.when(F.col("CustomerID").isNull(), "MISSING_CUSTOMER_ID"),
        F.when(F.col("_id_count") > 1, "DUPLICATE_CUSTOMER_ID"),
        F.when(~F.col("email").rlike(email_regex) & F.col("email").isNotNull(), "INVALID_EMAIL_FORMAT"),
        F.when(~F.col("Segment").isin("Consumer", "Corporate", "Home Office"), "INVALID_SEGMENT"),
        F.when(F.col("Country").isNull(), "MISSING_COUNTRY"),
        F.when(F.col("PostalCode").isNull(), "MISSING_POSTAL_CODE"),
# customer name checks
        F.when(F.col("CustomerName").isNull() | (F.trim(F.col("CustomerName")) == ""), "MISSING_CUSTOMER_NAME"),
        F.when(F.col("CustomerName").rlike(contains_digits), "NAME_CONTAINS_NUMBERS"),
        F.when(F.col("CustomerName").rlike(contains_bad_symbols), "NAME_CONTAINS_SPECIAL_CHARACTERS"),
        F.when(F.col("CustomerName").rlike(has_excess_spaces), "NAME_HAS_EXCESS_SPACES"),
        F.when(F.col("CustomerName") != F.trim(F.col("CustomerName")), "NAME_HAS_LEADING_TRAILING_SPACES"),

        # --- Phone Column Specific Rules ---
        F.when(F.col("phone").isNull() | (F.trim(F.col("phone")) == ""), "MISSING_PHONE"),
        F.when(F.col("phone").rlike(is_system_error), "PHONE_HAS_SYSTEM_ERROR"),
        F.when(F.col("phone").rlike(is_negative_number), "PHONE_IS_NEGATIVE_NUMBER"),
        F.when(F.col("phone").rlike(has_phone_extension), "PHONE_HAS_EXTENSION_SUFFIX"),
        F.when(~F.col("phone").rlike(valid_phone_characters), "PHONE_INVALID_CHARACTERS"),
        F.when(F.col("phone").isNotNull() & ((F.length(clean_phone_digits) < 7) | (F.length(clean_phone_digits) > 15)), "PHONE_INVALID_DIGIT_COUNT")
    ]

    # 3. Append the error tracker array column and remove temporary window columns
    flagged_df = df_with_id_count.withColumn(
        "dq_errors", 
        F.array_compact(F.array(*error_conditions))
    ).drop("_id_count")

    # 4. Split data into Accepted (No errors) and Rejected (Has errors) DataFrames
    # A size of 0 means the row passed every rule perfectly
    accepted_df = flagged_df.filter(F.size(F.col("dq_errors")) == 0).drop("dq_errors")
    rejected_df = flagged_df.filter(F.size(F.col("dq_errors")) > 0)

    return accepted_df,rejected_df

Product Data Quality

In [0]:
def dqc_checks_product(prod_table_name):
    """
    Cleans and validates the product table data. 
    Bifurcates rows cleanly into Accepted and Rejected tables.
    """
    # -------------------------------------------------------------
    # STEP 1: PRE-CLEANSING & REFORMATTING
    # -------------------------------------------------------------
    # Standardize column headers by removing spaces and dashes
    # e.g., "Sub-Category" -> "SubCategory", "Priceperproduct" -> "Priceperproduct"
    
    df=spark.table(prod_table_name)
    df = df.filter(F.col("Priceperproduct").isNotNull() & F.trim(F.col("Priceperproduct")).rlike(r"^\d+(\.\d+)?$"))
    df=df.withColumn("Priceperproduct", F.col("Priceperproduct").cast("double"))
    rename_mapping = {col: col.replace(" ", "").replace("-", "") for col in df.columns}
    df_clean = df.withColumnsRenamed(rename_mapping)

    
    # -------------------------------------------------------------
    # STEP 2: DEFINE QUALITY VALIDATION RULES
    # -------------------------------------------------------------
    # Regex rules to catch text damage
    contains_digits = r"[0-9]"
    contains_bad_symbols = r"[^A-Za-z\s\-\']"
    
    # Evaluate unique product keys using a temporary analytical window block
    df_with_id_count = df_clean.withColumn(
        "_id_count", 
        F.count("ProductID").over(Window.partitionBy("ProductID"))
    )

    error_conditions = [
        # --- ProductID Rules ---
        F.when(F.col("ProductID").isNull() | (F.trim(F.col("ProductID")) == ""), "MISSING_PRODUCT_ID"),
        F.when(F.col("_id_count") > 1, "DUPLICATE_PRODUCT_ID"),
        
        # --- Category & Sub-Category Rules ---
        F.when(F.col("Category").isNull() | (F.trim(F.col("Category")) == ""), "MISSING_CATEGORY"),
        # F.when(F.col("Category").rlike(contains_digits), "CATEGORY_CONTAINS_NUMBERS"),
        F.when(F.col("SubCategory").isNull() | (F.trim(F.col("SubCategory")) == ""), "MISSING_SUB_CATEGORY"),
        
        # --- ProductName Rules ---
        F.when(F.col("ProductName").isNull() | (F.trim(F.col("ProductName")) == ""), "MISSING_PRODUCT_NAME"),
        
        # --- State Rules ---
        F.when(F.col("State").isNull() | (F.trim(F.col("State")) == ""), "MISSING_STATE"),
        F.when(F.col("State").rlike(contains_digits), "STATE_CONTAINS_NUMBERS"),
        F.when(F.col("State").rlike(contains_bad_symbols), "STATE_CONTAINS_SPECIAL_CHARACTERS"),
        
        # --- Priceperproduct Rules ---
        F.when(F.col("Priceperproduct").isNull(), "MISSING_PRICE"),
        F.when(F.col("Priceperproduct").isNotNull() & ~F.col("Priceperproduct").rlike(r"^\d+(\.\d+)?$"), 
    "PRICE_CONTAINS_NON_DIGITS"),
        F.when(F.col("Priceperproduct") <= 0, "INVALID_PRICE_ZERO_OR_NEGATIVE")
    ]

    flagged_df = df_with_id_count.withColumn(
        "dq_errors", 
        F.concat_ws(",", F.array_compact(F.array(*error_conditions)))
    ).drop("_id_count")

    # Split the records based on the text error logs
    accepted_df = flagged_df.filter(F.col("dq_errors") == "").drop("dq_errors")
    rejected_df = flagged_df.filter(F.col("dq_errors") != "")
    return accepted_df,rejected_df


Orders Data Quality

In [0]:
def dqc_checks_orders(order_table_name):
    df = spark.table(order_table_name)
    valid_ship_modes = ["Standard Class", "Second Class", "First Class", "Same Day"]

    # 1. FIX: Use F.try_to_date instead of F.to_date to gracefully return NULL on malformed text
    df_parsed = df.withColumn("parsed_order_dt", F.try_to_date(F.col("OrderDate"), "d/M/yyyy")) \
                  .withColumn("parsed_ship_dt", F.try_to_date(F.col("ShipDate"), "d/M/yyyy"))

    df_with_id_count = df_parsed.withColumn(
        "_id_count", 
        F.count("RowID").over(Window.partitionBy("RowID"))
    )

    # 2. The rest of your conditions remain exactly the same
    error_conditions = [
        F.when(F.col("RowID").isNull(), "MISSING_ROW_ID"),
        F.when(F.col("_id_count") > 1, "DUPLICATE_ROW_ID"),
        F.when(F.col("OrderID").isNull() | (F.trim(F.col("OrderID")) == ""), "MISSING_ORDER_ID"),
        F.when(F.col("CustomerID").isNull() | (F.trim(F.col("CustomerID")) == ""), "MISSING_CUSTOMER_ID"),
        F.when(F.col("ProductID").isNull() | (F.trim(F.col("ProductID")) == ""), "MISSING_PRODUCT_ID"),
        
        # This will now work perfectly because try_to_date will make parsed_order_dt NULL
        F.when(F.col("OrderDate").isNull(), "MISSING_ORDER_DATE"),
        F.when(F.col("ShipDate").isNull(), "MISSING_SHIP_DATE"),
        F.when(F.col("parsed_order_dt").isNull() & F.col("OrderDate").isNotNull(), "MALFORMED_ORDER_DATE_FORMAT"),
        F.when(F.col("parsed_ship_dt").isNull() & F.col("ShipDate").isNotNull(), "MALFORMED_SHIP_DATE_FORMAT"),
        F.when(F.col("parsed_order_dt") > F.col("parsed_ship_dt"), "LOGICAL_ERROR_ORDER_DATE_AFTER_SHIP_DATE"),
        
        # Numeric Boundary Checks
        F.when(F.col("Quantity").isNull() | (F.col("Quantity") <= 0), "INVALID_QUANTITY_VALUE"),
        F.when(F.col("Price").isNull() | (F.col("Price") <= 0), "INVALID_PRICE_VALUE"),
        F.when(F.col("Discount").isNull() | (F.col("Discount") < 0) | (F.col("Discount") > 1), "DISCOUNT_OUT_OF_BOUNDS"),
        F.when(F.col("Profit").isNull(), "MISSING_PROFIT_VALUE"),
        
        # Categorical Validity Check
        F.when(F.col("ShipMode").isNull() | (F.trim(F.col("ShipMode")) == ""), "MISSING_SHIP_MODE"),
        F.when(~F.col("ShipMode").isin(valid_ship_modes) & F.col("ShipMode").isNotNull(), "INVALID_SHIP_MODE_CATEGORY")
    ]

    # 3. Append error tracker and drop ALL temporary columns
    flagged_df = df_with_id_count.withColumn(
        "dq_errors", 
        F.array_compact(F.array(*error_conditions))
    ).drop("_id_count", "parsed_order_dt", "parsed_ship_dt")

    # 4. Split data into Accepted and Rejected
    accepted_df = flagged_df.filter(F.size(F.col("dq_errors")) == 0).drop("dq_errors")
    rejected_df = flagged_df.filter(F.size(F.col("dq_errors")) > 0)

    return accepted_df, rejected_df

DQC Write

In [0]:
def dqc_checks_write(input_table,target_table_accepted,target_table_rejected):
    # Idempotently write records out using overwrite mode with schema mapping safety
    if input_table=="assessment.bronze.customer":
        accepted_df,rejected_df=dqc_checks_customer(input_table)
    elif input_table=="assessment.bronze.product":
        accepted_df,rejected_df=dqc_checks_product(input_table)
    elif input_table=="assessment.bronze.orders":
        accepted_df,rejected_df=dqc_checks_orders(input_table)
    print(f"Saving accepted record for input {input_table}...")
    accepted_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table_accepted)

    print(f"Saving rejected records...")
    rejected_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table_rejected)
        
    print("Data quality pipeline complete.")